# ROGII - test U-continuity fade (seen in a public competitor notebook) on our sp45

Isolated diagnostic on true_holdout_160 -- no submission. Sweeps cap/tau around the competitor-observed values (cap=8, tau=240).

Run: Input = competition dataset + rogii-gru-v12-assets (private). CPU. Internet off. Run All.


In [ ]:
"""Test the 'U-continuity fade' post-processor (seen in a public competitor notebook,
rogii-exact-public-6-768-frontier-visuals.ipynb, whose live score improved 6.768->6.703 after adding
it) on our OWN sp45 track, honest true_holdout_160 pooled RMSE, no submission. Mechanism: at the
known/hidden boundary, compute gap_u = pred[first_hidden] + Z[first_hidden] - (TVT_input[last_known] +
Z[last_known]) in the U=TVT+Z structural-elevation coordinate (the same coordinate our own pf_z/beam2
use internally), then apply move = -clip(gap_u, -cap, cap) * exp(-md_since/tau), added directly to the
TVT prediction. This patches the boundary DISCONTINUITY specifically (decays to ~0 by md_since>>tau),
unlike our own physics_postprocess which fits a global degree-4 trend + warmup-ramps a blend (opposite
shape: theirs is strong at the boundary and fades OUT; ours is weak at the boundary and ramps IN).
Caveat (flagged before running): this is an ISOLATED sp45-level test, like many earlier diagnostics in
this project (denoise, wall-hedge, WARP-blend) -- a real signal here is a green light to build a real
submission notebook, not a guarantee it survives contact with the full pipeline (file 33 today showed
even a strong isolated real-3-wells win can still fail once composed with the rest of the stack).
"""
import glob, os, pickle, numpy as np, pandas as pd

_c = glob.glob('/kaggle/input/**/*__horizontal_well.csv', recursive=True)
_t = [p for p in _c if 'train' in p.lower()]
_c = _t if _t else _c
TRAIN_DIR = os.path.dirname(_c[0]) if _c else 'd:/ROGII/data/train'
_assets = glob.glob('/kaggle/input/**/proxy_slim.pkl', recursive=True)
ASSET_DIR = os.path.dirname(_assets[0]) if _assets else '.'
print(f'TRAIN_DIR={TRAIN_DIR}  ASSET_DIR={ASSET_DIR}', flush=True)

proxy = pickle.load(open(f'{ASSET_DIR}/proxy_slim.pkl', 'rb'))
holdout_160 = pickle.load(open(f'{ASSET_DIR}/warp_true_holdout_160.pkl', 'rb'))

CAP_TAU_GRID = [(8.0, 240.0), (4.0, 240.0), (16.0, 240.0), (8.0, 120.0), (8.0, 480.0), (12.0, 300.0)]

results = {cap_tau: [] for cap_tau in CAP_TAU_GRID}
sp45_sqerr = []
n_total = 0

for wid in sorted(holdout_160):
    if wid not in proxy: continue
    hw = pd.read_csv(f'{TRAIN_DIR}/{wid}__horizontal_well.csv')
    km = hw['TVT_input'].notna()
    if km.sum() < 20: continue
    known_idx = np.flatnonzero(km.values)
    last_known = int(known_idx[-1])
    MD = hw['MD'].values.astype(float)
    Z = hw['Z'].values.astype(float)
    TVT_input = hw['TVT_input'].values.astype(float)
    last_u = float(TVT_input[last_known] + Z[last_known])
    last_md = float(MD[last_known])

    px = proxy[wid]
    pred = px['sp45'].astype(np.float64).copy()
    true = px['true'].astype(np.float64)
    md_eval = px['md'].astype(np.float64)
    z_eval = px['z'].astype(np.float64)
    n = len(pred)
    if n == 0: continue

    md_since = md_eval - last_md
    if md_since[0] <= 0:
        continue  # shouldn't happen; guard against bad alignment

    sp45_sqerr.append((pred - true) ** 2)
    n_total += n

    gap_u = float(pred[0] + z_eval[0] - last_u)
    for cap, tau in CAP_TAU_GRID:
        move = -np.clip(gap_u, -cap, cap) * np.exp(-md_since / tau)
        refined = pred + move
        results[(cap, tau)].append((refined - true) ** 2)

sp45_rmse = float(np.sqrt(np.sum(np.concatenate(sp45_sqerr)) / n_total))
print(f'sp45 baseline pooled RMSE (true_holdout_160, n_rows={n_total}): {sp45_rmse:.4f}', flush=True)
print(flush=True)
for cap_tau, sqs in results.items():
    r = float(np.sqrt(np.sum(np.concatenate(sqs)) / n_total))
    delta = r - sp45_rmse
    print(f'cap={cap_tau[0]:5.1f} tau={cap_tau[1]:5.1f}  pooled_rmse={r:.4f}  delta_vs_sp45={delta:+.4f}', flush=True)

